## 1. Mount Google Drive & Imports
Mount Drive and import all required libraries for training.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import time
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.model_selection import train_test_split
from PIL import Image

print("All imports done.")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load Pipeline Metadata
Load model configuration and hierarchy mapping saved by data.ipynb.

In [ ]:
pipeline_path = '/content/drive/MyDrive/xai_dataset/data_pipeline.pkl'

with open(pipeline_path, 'rb') as f:
    pipeline = pickle.load(f)

NUM_L1          = pipeline['NUM_L1']
NUM_L2          = pipeline['NUM_L2']
CONCEPT_NAMES   = pipeline['CONCEPT_NAMES']
attr_parent_idx = pipeline['attr_parent_idx']

print(f"NUM_L1        : {NUM_L1}")
print(f"NUM_L2        : {NUM_L2}")
print(f"CONCEPT_NAMES : {CONCEPT_NAMES}")


## 3. Build DataLoaders
Rebuild the full data pipeline from scratch — dataset paths, transforms, BirdDataset, and DataLoaders.


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────
dataset_dir = '/content/drive/MyDrive/xai_dataset/CUB_200_2011/CUB_200_2011/'
mask_root   = '/content/drive/MyDrive/xai_dataset/segmentations/'

IMG_SIZE   = 224
BATCH_SIZE = 32

# ── Transforms ─────────────────────────────────────────────────────────
train_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomCrop(IMG_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

eval_transform = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

mask_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
])

# ── Load raw files ──────────────────────────────────────────────────────
with open(os.path.join(dataset_dir, 'images.txt')) as f:
    image_list = [line.strip().split(' ')[1] for line in f]

with open(os.path.join(dataset_dir, 'image_class_labels.txt')) as f:
    labels = [int(line.strip().split(' ')[1]) for line in f]

attr_txt = os.path.join(dataset_dir, 'attributes.txt')
with open(attr_txt) as f:
    attr_names = [line.strip().split(' ', 1)[1] for line in f]

attr_label_txt = os.path.join(dataset_dir, 'attributes', 'image_attribute_labels.txt')
image_attr = {}
image_attr_certainty = {}
with open(attr_label_txt) as f:
    for line in f:
        parts = line.strip().split()
        img_id    = int(parts[0])
        attr_id   = int(parts[1]) - 1
        is_present = int(parts[2])
        certainty  = int(parts[3])
        image_attr.setdefault(img_id, {})[attr_id] = is_present
        image_attr_certainty.setdefault(img_id, {})[attr_id] = certainty

# ── Build attribute vectors ─────────────────────────────────────────────
num_attrs = len(attr_names)
attr_vectors = []
certainty_masks = []
for i in range(1, len(image_list) + 1):
    vec  = np.zeros(num_attrs, dtype=np.int8)
    cert = np.zeros(num_attrs, dtype=np.int8)
    for attr_id, is_present in image_attr.get(i, {}).items():
        vec[attr_id]  = is_present
        cert[attr_id] = 1 if image_attr_certainty.get(i, {}).get(attr_id, 0) >= 3 else 0
    attr_vectors.append(vec)
    certainty_masks.append(cert)

# ── Parse L1 groups ─────────────────────────────────────────────────────
MERGE_MAP = {
    'under_tail': 'tail', 'upper_tail': 'tail',
    'upperparts': 'back', 'underparts': 'belly',
}
NO_PARENT = {'primary', 'shape', 'size'}

part_to_attr_indices = {}
attr_to_part = {}
for idx, name in enumerate(attr_names):
    category = name.split('::')[0].replace('has_', '')
    for suffix in ('_color', '_pattern', '_shape', '_length'):
        if category.endswith(suffix):
            part = category[:len(category) - len(suffix)]
            break
    else:
        part = category
    part = MERGE_MAP.get(part, part)
    if part in NO_PARENT:
        attr_to_part[idx] = None
        continue
    attr_to_part[idx] = part
    part_to_attr_indices.setdefault(part, []).append(idx)

CONCEPT_NAMES = sorted(part_to_attr_indices.keys())
part_to_idx   = {p: i for i, p in enumerate(CONCEPT_NAMES)}

# ── Visibility ──────────────────────────────────────────────────────────
CUB_PART_TO_L1 = {
    1:'back', 2:'bill', 3:'belly', 4:'breast', 5:'crown',
    6:'forehead', 7:'eye', 8:'leg', 9:'wing', 10:'nape',
    11:'eye', 12:'leg', 13:'wing', 14:'tail', 15:'throat',
}
part_locs_txt = os.path.join(dataset_dir, 'parts', 'part_locs.txt')
image_part_visible = {}
with open(part_locs_txt) as f:
    for line in f:
        cols = line.strip().split()
        img_id = int(cols[0]); cub_part_id = int(cols[1]); visible = int(cols[4])
        l1_part = CUB_PART_TO_L1.get(cub_part_id)
        if l1_part and l1_part in part_to_idx:
            image_part_visible.setdefault(img_id, {})
            l1_idx = part_to_idx[l1_part]
            image_part_visible[img_id][l1_idx] = max(
                image_part_visible[img_id].get(l1_idx, 0), visible)

if 'head' in part_to_idx:
    head_idx  = part_to_idx['head']
    sub_idxs  = [part_to_idx[s] for s in ['crown','forehead','eye','nape'] if s in part_to_idx]
    for img_id in image_part_visible:
        image_part_visible[img_id][head_idx] = max(
            image_part_visible[img_id].get(si, 0) for si in sub_idxs)

# ── Splits ──────────────────────────────────────────────────────────────
image_ids = list(range(1, len(image_list) + 1))
with open(os.path.join(dataset_dir, 'train_test_split.txt')) as f:
    is_train = [int(line.strip().split(' ')[1]) for line in f]

all_train_images = [img for img,t in zip(image_list,     is_train) if t==1]
all_train_labels = [lbl for lbl,t in zip(labels,         is_train) if t==1]
all_train_attrs  = [vec for vec,t in zip(attr_vectors,   is_train) if t==1]
all_train_certs  = [c   for c,  t in zip(certainty_masks,is_train) if t==1]
all_train_ids    = [iid for iid,t in zip(image_ids,      is_train) if t==1]

test_images = [img for img,t in zip(image_list,     is_train) if t==0]
test_labels = [lbl for lbl,t in zip(labels,         is_train) if t==0]
test_attrs  = [vec for vec,t in zip(attr_vectors,   is_train) if t==0]
test_certs  = [c   for c,  t in zip(certainty_masks,is_train) if t==0]
test_ids    = [iid for iid,t in zip(image_ids,      is_train) if t==0]

indices = list(range(len(all_train_images)))
train_idx, val_idx = train_test_split(
    indices, test_size=0.1, random_state=42, stratify=all_train_labels)

train_images = [all_train_images[i] for i in train_idx]
train_labels = [all_train_labels[i] for i in train_idx]
train_attrs  = [all_train_attrs[i]  for i in train_idx]
train_certs  = [all_train_certs[i]  for i in train_idx]
train_ids    = [all_train_ids[i]    for i in train_idx]

val_images   = [all_train_images[i] for i in val_idx]
val_labels   = [all_train_labels[i] for i in val_idx]
val_attrs    = [all_train_attrs[i]  for i in val_idx]
val_certs    = [all_train_certs[i]  for i in val_idx]
val_ids      = [all_train_ids[i]    for i in val_idx]

# ── L1/L2/vis arrays ───────────────────────────────────────────────────
def build_l1(attr_vecs):
    result = np.zeros((len(attr_vecs), NUM_L1), dtype=np.float32)
    for i, vec in enumerate(attr_vecs):
        for part, idxs in part_to_attr_indices.items():
            if any(vec[j] for j in idxs):
                result[i, part_to_idx[part]] = 1.0
    return result

def build_vis(img_id_list):
    result = np.zeros((len(img_id_list), NUM_L1), dtype=np.float32)
    for i, iid in enumerate(img_id_list):
        for l1_idx, v in image_part_visible.get(iid, {}).items():
            result[i, l1_idx] = float(v)
    return result

train_l1 = build_l1(train_attrs); train_l2 = np.array(train_attrs, dtype=np.float32)
train_cm = np.array(train_certs,  dtype=np.float32); train_vis = build_vis(train_ids)

val_l1   = build_l1(val_attrs);   val_l2   = np.array(val_attrs,   dtype=np.float32)
val_cm   = np.array(val_certs,    dtype=np.float32); val_vis   = build_vis(val_ids)

test_l1  = build_l1(test_attrs);  test_l2  = np.array(test_attrs,  dtype=np.float32)
test_cm  = np.array(test_certs,   dtype=np.float32); test_vis  = build_vis(test_ids)

# ── BirdDataset ─────────────────────────────────────────────────────────
class BirdDataset(Dataset):
    def __init__(self, image_paths, labels, l1, l2, cert, vis,
                 img_root, mask_root, transform=None, mask_transform=None):
        self.paths=image_paths; self.labels=labels
        self.l1=l1; self.l2=l2; self.cert=cert; self.vis=vis
        self.img_root=img_root; self.mask_root=mask_root
        self.transform=transform; self.mask_transform=mask_transform

    def __len__(self): return len(self.paths)

    def __getitem__(self, idx):
        img = Image.open(os.path.join(self.img_root, 'images', self.paths[idx])).convert('RGB')
        if self.transform: img = self.transform(img)
        mask_path = os.path.join(self.mask_root,
                                 os.path.splitext(self.paths[idx])[0] + '.png')
        if os.path.exists(mask_path):
            mask = Image.open(mask_path).convert('L')
            if self.mask_transform: mask = self.mask_transform(mask)
        else:
            mask = torch.zeros(1, IMG_SIZE, IMG_SIZE)
        label = torch.tensor(self.labels[idx] - 1, dtype=torch.long)
        l1    = torch.tensor(self.l1[idx],   dtype=torch.float32)
        l2    = torch.tensor(self.l2[idx],   dtype=torch.float32)
        cert  = torch.tensor(self.cert[idx], dtype=torch.float32)
        vis   = torch.tensor(self.vis[idx],  dtype=torch.float32)
        return img, label, l1, l2, mask, cert, vis

train_dataset = BirdDataset(train_images, train_labels, train_l1, train_l2,
                             train_cm, train_vis, dataset_dir, mask_root,
                             train_transform, mask_transform)
val_dataset   = BirdDataset(val_images,   val_labels,   val_l1,   val_l2,
                             val_cm,   val_vis,   dataset_dir, mask_root,
                             eval_transform, mask_transform)
test_dataset  = BirdDataset(test_images,  test_labels,  test_l1,  test_l2,
                             test_cm,  test_vis,  dataset_dir, mask_root,
                             eval_transform, mask_transform)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

print(f"Train batches : {len(train_loader)}")
print(f"Val batches   : {len(val_loader)}")
print(f"Test batches  : {len(test_loader)}")

## 4. Model Definition — HierarchicalCBM
Import from model.ipynb if already run in this session, otherwise define here.

In [ ]:
try:
    
    _ = HierarchicalCBM
    print("HierarchicalCBM already defined — using existing class.")
except NameError:

    class HierarchicalCBM(nn.Module):
        """
        Hierarchical Concept Bottleneck Model (H-CBM).

        Backbone   : ResNet-50 (pretrained ImageNet) → Feature Map F (2048,)
        Coarse Head: g_c(F)      → p_c (NUM_L1,)  — coarse part probabilities
        Fine Head  : g_f(F, p_c) → p_f (NUM_L2,)  — masked fine attribute probabilities
        Classifier : h(p_f)      → (num_classes,)  — reads ONLY from p_f
        """

        def __init__(self, attr_parent_idx, num_classes=200, num_l1=13, num_l2=312):
            super().__init__()
            self.num_l1 = num_l1
            self.num_l2 = num_l2
            self.register_buffer('attr_parent_idx',
                                 torch.tensor(attr_parent_idx, dtype=torch.long))

            # Backbone
            backbone = resnet50(weights=ResNet50_Weights.IMAGENET1K_V1)
            self.features = nn.Sequential(*list(backbone.children())[:-1])

            # Coarse Head g_c(F)
            self.coarse_head = nn.Sequential(
                nn.Linear(2048, 512), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(512, num_l1),
            )

            # Fine Head g_f(F, p_c)
            self.fine_head = nn.Sequential(
                nn.Linear(2048 + num_l1, 512), nn.ReLU(), nn.Dropout(0.3),
                nn.Linear(512, num_l2),
            )

            # Classifier h(p_f) — reads ONLY from p_f
            self.classifier = nn.Linear(num_l2, num_classes)

        def forward(self, x):
            feats   = self.features(x).flatten(1)              # (B, 2048)
            z_c     = self.coarse_head(feats)                  # (B, 13)
            p_c     = torch.sigmoid(z_c)                       # (B, 13)

            z_f     = self.fine_head(torch.cat([feats, p_c], dim=1))
            p_f_raw = torch.sigmoid(z_f)

            parent_idx   = self.attr_parent_idx
            safe_idx     = parent_idx.clamp(min=0)
            parent_probs = p_c[:, safe_idx]
            mask = torch.where(
                (parent_idx >= 0).unsqueeze(0),
                parent_probs,
                torch.ones_like(parent_probs),
            )
            p_f = p_f_raw * mask

            cls_logits = self.classifier(p_f)
            return cls_logits, p_c, p_f, z_c, z_f, feats 

    print("HierarchicalCBM defined in this session.")

## 5. Instantiate Model & Select Device
Load model config, instantiate HierarchicalCBM, and move to GPU.

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")
if device.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")

model = HierarchicalCBM(
    attr_parent_idx = attr_parent_idx,
    num_classes     = 200,
    num_l1          = NUM_L1,
    num_l2          = NUM_L2,
).to(device)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTotal parameters     : {total_params:,}")
print(f"Trainable parameters : {trainable_params:,}")

## 6. Loss Functions
Define Focal Loss for L2 attributes, Weighted BCE for L1 parts, and CrossEntropy for species classification.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for binary classification with class imbalance.
    FL(p_t) = -alpha * (1 - p_t)^gamma * log(p_t)
    
    Used for L2 attributes — 312 binary attributes with severe imbalance.
    gamma=2 focuses training on hard/misclassified examples.
    """
    def __init__(self, alpha=0.25, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets, mask=None):
        """
        Args:
            logits  : (B, 312) — raw logits from Fine Head
            targets : (B, 312) — binary ground truth
            mask    : (B, 312) — 1 where loss should be computed, 0 elsewhere
                      combines certainty mask AND visibility mask
        """
        bce  = F.binary_cross_entropy_with_logits(logits, targets, reduction='none')
        p_t  = torch.exp(-bce)
        loss = self.alpha * (1 - p_t) ** self.gamma * bce

        if mask is not None:
            loss = loss * mask
            denom = mask.sum().clamp(min=1)
            return loss.sum() / denom

        return loss.mean()


def weighted_bce_loss(logits, targets, mask=None, pos_weight=None):
    """
    Weighted BCE for L1 coarse concept loss.
    
    Args:
        logits     : (B, 13) — raw logits from Coarse Head
        targets    : (B, 13) — binary ground truth
        mask       : (B, 13) — visibility mask (1 = part visible in image)
        pos_weight : (13,)   — per-part positive class weight
    """
    loss = F.binary_cross_entropy_with_logits(
        logits, targets, pos_weight=pos_weight, reduction='none'
    )

    if mask is not None:
        loss = loss * mask
        denom = mask.sum().clamp(min=1)
        return loss.sum() / denom

    return loss.mean()


# ── Compute pos_weight for L1 from training data ──────────────────────
# pos_weight[i] = N_negative / N_positive for part i
l1_pos  = train_l1.sum(axis=0)                          # (13,)
l1_neg  = len(train_l1) - l1_pos                        # (13,)
l1_pos_weight = torch.tensor(
    l1_neg / np.clip(l1_pos, 1, None), dtype=torch.float32
).to(device)

focal_loss_fn = FocalLoss(alpha=0.25, gamma=2.0)

print("Loss functions defined.")
print(f"L1 pos_weight range: [{l1_pos_weight.min():.2f}, {l1_pos_weight.max():.2f}]")

## 7. Optimizer & Scheduler
AdamW with different learning rates for backbone vs heads. ReduceLROnPlateau scheduler.

In [ ]:
# Separate parameter groups — lower lr for backbone to prevent catastrophic forgetting
backbone_params = list(model.features.parameters())
head_params     = (list(model.coarse_head.parameters()) +
                   list(model.fine_head.parameters()) +
                   list(model.classifier.parameters()))

optimizer = torch.optim.AdamW([
    {'params': backbone_params, 'lr': 1e-4},
    {'params': head_params,     'lr': 1e-3},
], weight_decay=1e-4)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', patience=5, factor=0.5
)

# Loss weights — fixed
LAMBDA_C   = 0.5   # L_coarse weight
LAMBDA_F   = 0.5   # L_fine weight
LAMBDA_CLS = 1.0   # L_task weight

print("Optimizer and scheduler defined.")
print(f"Backbone lr : 1e-4")
print(f"Heads lr    : 1e-3")
print(f"Lambda weights — coarse: {LAMBDA_C}, fine: {LAMBDA_F}, task: {LAMBDA_CLS}")

## 8. Training Loop
Full training loop with validation, early stopping, and checkpoint saving.

In [ ]:
def compute_losses(model, imgs, lbls, l1s, l2s, certs, viss, device):
    """Single forward pass — all three losses computed from one backbone call."""
    cls_logits, p_c, p_f, z_c, z_f, feats = model(imgs)

    # ── L_coarse — masked by visibility ───────────────────────────────
    l_coarse = weighted_bce_loss(
        logits=z_c, targets=l1s,
        mask=viss, pos_weight=l1_pos_weight,
    )

    # ── L_fine — masked by certainty × parent visibility ──────────────
    parent_idx = model.attr_parent_idx          # (312,)
    safe_idx   = parent_idx.clamp(min=0)
    parent_vis = viss[:, safe_idx]              # (B, 312)
    has_parent = (parent_idx >= 0).unsqueeze(0) # (1, 312)
    vis_mask   = torch.where(has_parent, parent_vis, torch.ones_like(parent_vis))
    l2_mask    = certs * vis_mask               # (B, 312)

    l_fine = focal_loss_fn(logits=z_f, targets=l2s, mask=l2_mask)

    # ── L_task ─────────────────────────────────────────────────────────
    l_task = F.cross_entropy(cls_logits, lbls)

    loss = LAMBDA_C * l_coarse + LAMBDA_F * l_fine + LAMBDA_CLS * l_task
    return loss, l_coarse.item(), l_fine.item(), l_task.item(), cls_logits


def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total_loss = 0.0
    correct    = 0
    total      = 0

    for imgs, lbls, l1s, l2s, _, certs, viss in loader:
        imgs  = imgs.to(device);  lbls  = lbls.to(device)
        l1s   = l1s.to(device);   l2s   = l2s.to(device)
        certs = certs.to(device); viss  = viss.to(device)

        optimizer.zero_grad()
        loss, _, _, _, cls_logits = compute_losses(
            model, imgs, lbls, l1s, l2s, certs, viss, device)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item()
        correct    += (cls_logits.argmax(dim=1) == lbls).sum().item()
        total      += lbls.size(0)

    return total_loss / len(loader), correct / total


def evaluate(model, loader, device):
    model.eval()
    total_loss = 0.0
    correct    = 0
    total      = 0

    with torch.no_grad():
        for imgs, lbls, l1s, l2s, _, certs, viss in loader:
            imgs  = imgs.to(device);  lbls  = lbls.to(device)
            l1s   = l1s.to(device);   l2s   = l2s.to(device)
            certs = certs.to(device); viss  = viss.to(device)

            loss, _, _, _, cls_logits = compute_losses(
                model, imgs, lbls, l1s, l2s, certs, viss, device)

            total_loss += loss.item()
            correct    += (cls_logits.argmax(dim=1) == lbls).sum().item()
            total      += lbls.size(0)

    return total_loss / len(loader), correct / total


print("compute_losses, train_one_epoch, evaluate defined.")

## 9. Run Training
Execute the training loop with early stopping and best model checkpointing.

In [ ]:
CHECKPOINT_PATH = '/content/drive/MyDrive/xai_dataset/best_model.pt'
MAX_EPOCHS      = 50
PATIENCE        = 10

best_val_loss    = float('inf')
patience_counter = 0
history          = {'train_loss': [], 'val_loss': [],
                    'train_acc':  [], 'val_acc':  []}

print(f"Starting training — max {MAX_EPOCHS} epochs, patience={PATIENCE}")
print(f"Checkpoint: {CHECKPOINT_PATH}\n")

for epoch in range(1, MAX_EPOCHS + 1):
    t0 = time.time()

    train_loss, train_acc = train_one_epoch(model, train_loader, optimizer, device)
    val_loss,   val_acc   = evaluate(model, val_loader, device)

    scheduler.step(val_loss)

    history['train_loss'].append(train_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(train_acc)
    history['val_acc'].append(val_acc)

    elapsed    = time.time() - t0
    current_lr = optimizer.param_groups[1]['lr']
    print(f"Epoch {epoch:3d}/{MAX_EPOCHS} | "
          f"train_loss: {train_loss:.4f} | train_acc: {train_acc:.4f} | "
          f"val_loss: {val_loss:.4f} | val_acc: {val_acc:.4f} | "
          f"lr: {current_lr:.2e} | time: {elapsed:.1f}s")

    # ── Checkpoint ─────────────────────────────────────────────────────
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        patience_counter = 0
        torch.save({
            'epoch':            epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state':  optimizer.state_dict(),
            'val_loss':         best_val_loss,
            'val_acc':          val_acc,
            'NUM_L1':           NUM_L1,
            'NUM_L2':           NUM_L2,
            'CONCEPT_NAMES':    CONCEPT_NAMES,
            'attr_parent_idx':  attr_parent_idx,
        }, CHECKPOINT_PATH)
        print(f"  ✅ Best model saved (val_loss={best_val_loss:.4f})")
    else:
        patience_counter += 1
        print(f"  ⏳ No improvement ({patience_counter}/{PATIENCE})")
        if patience_counter >= PATIENCE:
            print(f"\nEarly stopping at epoch {epoch}.")
            break

print(f"\nTraining complete. Best val_loss: {best_val_loss:.4f}")

## 10. Plot Training History
Visualize loss and accuracy curves after training.

In [ ]:
import matplotlib.pyplot as plt

epochs_ran = len(history['train_loss'])
x = range(1, epochs_ran + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(x, history['train_loss'], label='Train Loss')
ax1.plot(x, history['val_loss'],   label='Val Loss')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
ax1.set_title('Loss Curve'); ax1.legend(); ax1.grid(True)

ax2.plot(x, history['train_acc'], label='Train Acc')
ax2.plot(x, history['val_acc'],   label='Val Acc')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Accuracy')
ax2.set_title('Accuracy Curve'); ax2.legend(); ax2.grid(True)

plt.tight_layout()
plt.savefig('/content/drive/MyDrive/xai_dataset/training_curves.png', dpi=150)
plt.show()
print("Saved to Drive.")

## 11. Training Roadmap
Summary of what happened in each section of this notebook.

# Training Pipeline Roadmap

**Phase 3 of 5** in the XAI project pipeline.

```
data.ipynb  ✅  →  model.ipynb  ✅  →  train.ipynb  ✅  →  evaluate.ipynb  ⏳  →  xai.ipynb  ⏳
```

---

## Section 1 — Mount Google Drive & Imports
**Input:** Nothing.
**Process:** Mounts Drive and imports all required libraries.
**Output:** All libraries ready. Drive accessible at `/content/drive/MyDrive/`.

---

## Section 2 — Load Pipeline Metadata
**Input:** `data_pipeline.pkl` from Drive.
**Process:** Loads NUM_L1, NUM_L2, CONCEPT_NAMES, attr_parent_idx saved by data.ipynb.
**Output:**
```
NUM_L1=13, NUM_L2=312
CONCEPT_NAMES → ['back','belly','bill',...]
attr_parent_idx → np.array(312,)
```

---

## Section 3 — Build DataLoaders
**Input:** CUB-200-2011 files from Drive.
**Process:** Full data pipeline rebuild — loads images, attributes, certainty, visibility, builds L1/L2 labels, splits train/val/test, wraps in BirdDataset and DataLoader.
**Output:**
```
train_loader (169 batches)   val_loader (19 batches)   test_loader (182 batches)
Each batch: imgs(32,3,224,224), lbls(32,), l1s(32,13), l2s(32,312),
            masks(32,1,224,224), certs(32,312), viss(32,13)
```

---

## Section 4 — Model Definition — HierarchicalCBM
**Input:** Nothing — defines the class.
**Process:** Defines HierarchicalCBM. If already defined from model.ipynb in this session, reuses it. Otherwise defines it here.
**Output:** `HierarchicalCBM` class with updated forward that returns:
```
cls_logits (B,200), p_c (B,13), p_f (B,312), z_c (B,13), z_f (B,312), feats (B,2048)
```

---

## Section 5 — Instantiate Model & Select Device
**Input:** attr_parent_idx, NUM_L1, NUM_L2.
**Process:** Instantiates HierarchicalCBM and moves to GPU (T4).
**Output:**
```
model → ~25M parameters on cuda
device → cuda
```

---

## Section 6 — Loss Functions
**Input:** train_l1 for pos_weight computation.
**Process:** Defines two loss functions:
- `FocalLoss` for L2 — α=0.25, γ=2 — focuses on hard examples, handles class imbalance
- `weighted_bce_loss` for L1 — pos_weight = N_neg/N_pos per part
**Output:** `focal_loss_fn`, `weighted_bce_loss`, `l1_pos_weight (13,)` ready.

---

## Section 7 — Optimizer & Scheduler
**Input:** model parameters.
**Process:** AdamW with two parameter groups (backbone lr=1e-4, heads lr=1e-3). ReduceLROnPlateau scheduler.
**Output:**
```
optimizer → AdamW, weight_decay=1e-4
scheduler → ReduceLROnPlateau (patience=5, factor=0.5)
λ_c=0.5, λ_f=0.5, λ_cls=1.0
```

---

## Section 8 — Training Loop
**Input:** model, loaders, optimizer, device.
**Process:** Defines three functions:
- `compute_losses` — single forward pass, computes all three losses with correct masking
- `train_one_epoch` — runs one full epoch, backprop, gradient clipping (max_norm=1.0)
- `evaluate` — same losses without gradient, for validation

Key masking logic:
- L_coarse masked by `viss` — parts not visible → not penalized
- L_fine masked by `certs × vis_mask` — uncertain or invisible → not penalized
**Output:** Functions ready for training.

---

## Section 9 — Run Training
**Input:** model, loaders, optimizer, scheduler.
**Process:** Trains for max 50 epochs with early stopping (patience=10). Saves best checkpoint to Drive when val_loss improves.
**Output:**
```
best_model.pt → saved to Drive, contains:
  model_state_dict, optimizer_state
  epoch, val_loss, val_acc
  NUM_L1, NUM_L2, CONCEPT_NAMES, attr_parent_idx
```

---

## Section 10 — Plot Training History
**Input:** history dict with train/val loss and accuracy per epoch.
**Process:** Plots loss and accuracy curves side by side.
**Output:** `training_curves.png` saved to Drive.

---

**Next step → Phase 4: Evaluation** (`evaluate.ipynb`)
- Load best checkpoint from Drive
- Top-1 / Top-5 Accuracy on test set
- L1 Concept Accuracy, L2 macro-F1, AUROC
- Concept Intervention
- Hierarchical Distance

**Full pipeline:**
Phase 1 — data.ipynb      ✅ Data pipeline
Phase 2 — model.ipynb     ✅ Model architecture
Phase 3 — train.ipynb     ✅ Training loop
Phase 4 — evaluate.ipynb  ⏳ Metrics + Concept Intervention
Phase 5 — xai.ipynb       ⏳ Faithfulness + segmentation masks